# Lab 06 · Reference solution

The polished final implementation of [Lab 06: Agentic RAG from scratch](../README.md).

A complete agentic RAG stack in pure Python:

- Recursive chunker (paragraph → sentence → word, target 160 tokens, ~20% overlap).
- Numpy-backed vector index, sentence-transformers bi-encoder.
- `search_corpus` + `read_chunk` tools with structured errors and similarity floor.
- Agent loop with `MAX_STEPS = 8`, action-hash dedup, citations tracked by the loop.

> ⏱ Read time: ~8 min · Notebook ~22 cells.
> 📖 The lab walks through Step 7 (three failure modes deliberately) and
> Step 9 (swap to OpenAI embeddings). This solution ships the
> hardened implementation directly; come back to the lab for the
> diagnostic walk.

> 🔒 **Chunker config is pinned** (`TARGET_TOKENS=160`, `OVERLAP_TOKENS=32`).
> Lab 07-09 depend on the same chunk IDs; do not change without
> re-validating downstream labs.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Provider-agnostic chat client

In [ ]:
def chat_with_tools(messages: list[dict], tools: list[dict]) -> dict:
    """Call chat completion with tool schemas; return OpenAI-shaped assistant message."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name,
                 "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools, max_tokens=2048, temperature=0,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## Chunker

Recursive splitting: paragraph → sentence → word. Target 160 tokens
per chunk with ~32-token overlap. Tokens approximated as
`len(text.split()) / 0.75` — close enough for chunking decisions.

The chunker config matters for downstream stability: Labs 07-09's
metrics annotate against these chunk IDs. Don't change `TARGET_TOKENS`
or `OVERLAP_TOKENS` here without also updating the dependent labs.

In [ ]:
CORPUS_DIR = pathlib.Path("../corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]


def chunk_text(text: str,
               target_tokens: int = TARGET_TOKENS,
               overlap_tokens: int = OVERLAP_TOKENS) -> list[str]:
    """Recursive paragraph→sentence splitter targeting ~target_tokens per chunk."""
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approx_tokens(para)

        if para_tokens > target_tokens:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > target_tokens and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue

        if current_tokens + para_tokens > target_tokens and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append("\n\n".join(current))

    # Prepend tail of previous chunk for overlap
    if overlap_tokens <= 0 or len(chunks) < 2:
        return chunks

    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(overlap_tokens * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip() if tail else chunks[i])
    return overlapped


def first_heading(text: str) -> str:
    """Return the first markdown heading line, or empty string."""
    for line in text.split("\n"):
        if line.startswith("# "):
            return line[2:].strip()
    return ""


# Load corpus
all_chunks: list[dict] = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    title = first_heading(text)
    for i, body in enumerate(chunk_text(text)):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": body,
        })

print(f"Loaded {len({c['doc_id'] for c in all_chunks})} docs, "
      f"{len(all_chunks)} chunks")


**Sample output:**

```
Loaded 8 docs, 55 chunks
```

The 55-chunk count must match across the solution and the lab — Labs 07-09's chunk-ID annotations depend on this stability.

## Embedding index

`sentence-transformers/all-MiniLM-L6-v2` produces 384-dim embeddings on
CPU. `normalize_embeddings=True` is essential — it makes cosine
similarity equal a dot product, so retrieval is one matmul.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading bi-encoder...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

# Sanity: norms ~1.0 if normalized
norms = np.linalg.norm(embeddings, axis=1)
print(f"Embeddings: shape={embeddings.shape}, dtype={embeddings.dtype}")
print(f"Norms: min={norms.min():.4f}, max={norms.max():.4f}, mean={norms.mean():.4f}")


**Sample output:**

```
Loading bi-encoder...
Embeddings: shape=(55, 384), dtype=float32
Norms: min=1.0000, max=1.0000, mean=1.0000
```

## `search_corpus` + `read_chunk`

Two tools, structured-error envelope on each. The similarity floor
(`MIN_SIMILARITY=0.30`) is calibrated for MiniLM cosines — any chunk
below this is dropped, and if no chunk crosses the floor the result is
`status: empty`. Critical for the off-corpus query case.

In [ ]:
MIN_SIMILARITY = 0.30


def search_corpus(query: str, top_k: int = 5) -> dict:
    """Embed query; return top-k chunks by cosine; floor at MIN_SIMILARITY."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    try:
        query_emb = embedder.encode(
            [query.strip()],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
    except Exception as e:
        return {"status": "error", "kind": "embed_failure",
                "detail": f"{type(e).__name__}: {e}"}

    # Normalized vectors → dot product = cosine similarity
    scores = embeddings @ query_emb
    n = min(top_k, len(scores))
    top_indices = np.argsort(scores)[::-1][:n]
    top_scores = scores[top_indices]
    above_floor = top_scores >= MIN_SIMILARITY

    if not above_floor.any():
        return {
            "status": "empty", "query": query,
            "detail": (f"no chunks crossed similarity floor of {MIN_SIMILARITY} "
                       f"(top score was {top_scores.max():.3f})"),
        }

    results = []
    for idx, score in zip(top_indices[above_floor], top_scores[above_floor], strict=True):
        chunk = all_chunks[idx]
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(score),
        })
    return {"status": "ok", "results": results}


chunks_by_id = {c["chunk_id"]: c for c in all_chunks}


def read_chunk(chunk_id: str) -> dict:
    """Return full text of one chunk by ID."""
    if not chunk_id:
        return {"status": "error", "kind": "other", "detail": "empty chunk_id"}
    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {
            "status": "error", "kind": "not_found",
            "detail": f"no chunk with id {chunk_id!r}; valid IDs come from search_corpus results",
        }
    return {
        "status": "ok",
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "title": chunk["title"],
        "text": chunk["text"],
    }


## Tool schemas + agent loop

The loop pattern is identical to Lab 03's — same `_action_hash` dedup,
same step cap, same citation discipline (citations recorded on
successful `read_chunk`, not on `search_corpus` snippet view).

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_corpus",
            "description": (
                "Search the corpus by semantic similarity. Returns up to top_k "
                "chunks with title, snippet, and similarity score. Use this to "
                "triage which chunks are worth reading in full. Phrase queries "
                "as 3-8 specific words. Returns 'empty' if no chunk crosses the "
                "similarity floor."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "3-8 specific words."},
                    "top_k": {"type": "integer", "description": "1-10, default 5."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": (
                "Read the full text of one chunk. Use after search_corpus to "
                "inspect a candidate. Pass chunk_id exactly as it appeared in "
                "search_corpus results, e.g., '01-agent-loop.md:2'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {"type": "string", "description": "Chunk ID from search_corpus."},
                },
                "required": ["chunk_id"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        return search_corpus(query=args["query"], top_k=args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant grounded in a specific document
corpus. Answer the user's question only from the corpus.

Strategy:
1. Start with search_corpus using 3-8 specific words from the question.
2. Look at the snippets and similarity scores. If snippets answer the question,
   synthesize directly. If not, pick the 1-2 most relevant chunks and call
   read_chunk for their full text.
3. If results are empty or poor, refine the query and search again — but do
   not repeat queries you've already tried.
4. Stop when you can answer confidently with grounded evidence. Don't loop.
5. Cite the chunks you actually read. Do not cite snippets you only saw.

When the corpus doesn't contain the answer, say so plainly.
"""


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    """Run the RAG agent; returns answer + chunk-level citations."""
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)

        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {(msg['content'] or '')[:120]}...")
            return {
                "answer": msg["content"],
                "citations": citations,
                "steps": step,
                "stopped_reason": "answer_with_citations" if citations else "answer_without_read",
            }

        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)
            if ah in seen_actions:
                tool_result = {
                    "status": "error", "kind": "repeated_action",
                    "detail": f"You already called {tc['name']} with these arguments. Try a different query or chunk.",
                }
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED — refused]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    args_repr = (str(args)[:80] + "...") if len(str(args)) > 80 else str(args)
                    print(f"  → {tc['name']}({args_repr}) → {tool_result.get('status', '?')}")

                # Citations: only successful read_chunk calls
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })

            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": json.dumps(tool_result)[:4000],
            })

    return {
        "answer": (
            f"I reached the step limit ({max_steps}) without a confident answer. "
            f"I read {len(citations)} chunk(s)."
        ),
        "citations": citations,
        "steps": max_steps,
        "stopped_reason": "step_cap",
    }


## Demo

A query that requires multi-chunk synthesis. The agent searches,
reads one or two chunks in full, and synthesizes — with citations
recorded by the loop, not by the LLM.

In [ ]:
result = run_agent(
    "How does the agent loop track citations, and why is that better "
    "than letting the LLM record them?"
)
print(f"\n=== Answer ===\n{result['answer']}")
print(f"\n=== Citations ({len(result['citations'])}) ===")
for c in result["citations"]:
    print(f"  • [{c['chunk_id']}] {c['title']}")
print(f"\nSteps: {result['steps']}, stopped: {result['stopped_reason']}")


**Sample output (will vary):**

```
── Step 1 ──
  → search_corpus({'query': 'citation tracking agent loop'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '08-citation-tracking.md:0'}) → ok

── Step 3 ──
  ◆ FINAL: The agent loop appends a citation entry to the citations list every time read_chunk returns successfully...

=== Answer ===
The agent loop tracks citations as a structural property: every successful
read_chunk call appends (chunk_id, doc_id, title) to a citations list maintained
by the loop. This is better than letting the LLM record citations because the
LLM can hallucinate citations — claim it read sources it only saw in snippets,
or even invent URLs. The loop records what was *actually retrieved*, making
citations a verifiable property of the trajectory.

=== Citations (1) ===
  • [08-citation-tracking.md:0] Citation Tracking in Agentic Systems

Steps: 3, stopped: answer_with_citations
```

## Production readiness — out of scope here

For a real deployment add: a real vector store (Chroma, Qdrant, pgvector)
for persistence + ANN at scale, metadata filtering on chunk lookups,
batch indexing for large corpora, async tool execution, and structured
logging of `(query, top_k, scores, chunks_read)` for offline review.

For better retrieval quality, see Lab 07 (hybrid + reranking) and
Lab 08 (contextual augmentation + query rewriting). For measuring
retrieval quality, see Lab 09.